In [1]:
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path.home() / "fmri-fm/.env")

True

In [2]:
from functools import partial
from itertools import product
import json

import torch
import pandas as pd
from omegaconf import OmegaConf

from data.flat_data import FlatClipsDataset, make_flat_transform
import flat_mae.models_mae as models_mae
import flat_mae.masking as masking
import flat_mae.utils as ut

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [78]:
def make_loader(name: str):
    urls = {
        "hcp": "s3://medarc/fmri-fm/datasets/flat-clips/hcp-val-clips-16t",
        "nsd": "s3://medarc/fmri-fm/datasets/flat-clips/nsd-subj01-clips-16t",
        "ukbb": "s3://sophont/fmri-fm/datasets/flat-clips/ukbb-val-clips-mini-16t",
    }

    transform = make_flat_transform(img_size=(224, 560), normalize="frame")
    dataset = FlatClipsDataset(urls[name], transform=transform)

    mask_fn = masking.TubeMasking(
        mask_ratio=0.9,
        img_size=(224, 560),
        patch_size=16,
        num_frames=16,
        t_patch_size=4,
    )

    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=partial(masking.mask_collate, mask_fn=mask_fn),
        num_workers=8,
        drop_last=True,
    )
    return loader

In [79]:
def load_model(ckpt_path: Path, **kwargs):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    cfg = OmegaConf.create(ckpt["args"])
    model_kwargs = cfg.model_kwargs.copy()
    model_kwargs.update(kwargs)
    model = models_mae.__dict__[cfg.model](
        img_size=cfg.img_size,
        in_chans=cfg.in_chans,
        patch_size=cfg.patch_size,
        num_frames=cfg.num_frames,
        t_patch_size=cfg.t_patch_size,
        **model_kwargs,
    )
    model.load_state_dict(ckpt["model"])
    return model

In [80]:
@torch.inference_mode()
def evaluate(model, loader, device):
    torch.manual_seed(42)
    logger = ut.MetricLogger(delimiter="  ")
    for batch in logger.log_every(loader, print_freq=20, header="eval"):
        batch = ut.send_data(batch, device)

        x = batch["image"]
        mask = batch["img_mask"]
        visible_mask = batch["visible_mask"]

        with torch.autocast(device_type=device.type, dtype=torch.float16):
            loss = model(
                x, img_mask=mask, visible_mask=visible_mask, mask_ratio=None, with_state=False
            )
        logger.update(loss=loss.item())

    stats = {k: meter.global_avg for k, meter in logger.meters.items()}
    return stats

In [3]:
result_dir = Path("results/masked_recon")
result_dir.mkdir(exist_ok=True, parents=True)

keys = [
    "attn_reg-1",
    "cross_reg-1",
    "crossreg_reg-1",
    "crossreg_reg-4",
    "crossreg_reg-16",
    "attn_pep-4",
    "cross_pep-4",
    "crossreg_reg-4_pep-4",
]
peps = [0, 4]

In [ ]:
for ds in ["hcp", "nsd", "ukbb"]:
    loader = make_loader(ds)

    for key, pep in product(keys, peps):
        out_path = result_dir / f"{key}__pad-{pep}__{ds}.json"
        if out_path.exists():
            continue

        ckpt_path = f"checkpoints/decoders/{key}/pretrain/checkpoint-last.pth"
        model = load_model(ckpt_path, pred_edge_pad=pep)
        model.to(device)

        stats = evaluate(model, loader, device)
        record = {"dataset": ds, "key": key, "pad": pep, **stats}
        print(json.dumps(record))
        with out_path.open("w") as f:
            print(json.dumps(record), file=f)

eval  [ 0/81]  eta: 0:02:26  loss: 0.8318 (0.8318)  time: 1.8056  data: 1.7425  max mem: 4148
eval  [20/81]  eta: 0:00:11  loss: 0.8267 (0.8247)  time: 0.1116  data: 0.0499  max mem: 4148
eval  [40/81]  eta: 0:00:06  loss: 0.8243 (0.8211)  time: 0.1232  data: 0.0608  max mem: 4148
eval  [60/81]  eta: 0:00:02  loss: 0.8265 (0.8212)  time: 0.1101  data: 0.0475  max mem: 4148
eval  [80/81]  eta: 0:00:00  loss: 0.8302 (0.8231)  time: 0.1164  data: 0.0553  max mem: 4148
eval Total time: 0:00:11 (0.1381 s / it)
{"dataset": "hcp", "key": "attn_reg-1", "pad": 0, "loss": 0.8231088305697029}
eval  [ 0/81]  eta: 0:02:29  loss: 0.8771 (0.8771)  time: 1.8402  data: 1.7519  max mem: 4148
eval  [20/81]  eta: 0:00:11  loss: 0.8719 (0.8697)  time: 0.1111  data: 0.0457  max mem: 4148
eval  [40/81]  eta: 0:00:06  loss: 0.8699 (0.8660)  time: 0.1237  data: 0.0577  max mem: 4148
eval  [60/81]  eta: 0:00:03  loss: 0.8706 (0.8662)  time: 0.1116  data: 0.0464  max mem: 4148
eval  [80/81]  eta: 0:00:00  loss: 

In [4]:
records = []

for ds in ["hcp", "nsd", "ukbb"]:
    for key, pep in product(keys, peps):
        out_path = result_dir / f"{key}__pad-{pep}__{ds}.json"
        with out_path.open() as f:
            record = json.load(f)
        records.append(record)

df = pd.DataFrame.from_records(records)

In [7]:
print(df.shape)
df.head()
df.to_csv("results/masked_recon.csv", index=False)

(48, 4)


In [ ]:
subdf = df.pivot_table(values="loss", index="key", columns=["dataset", "pad"], sort=False)
subdf = subdf.iloc[[0, 1, 4, 3, 2, 5, 6, 7], [1, 3, 5]]
subdf = subdf.reset_index()
subdf.columns = ["key", "HCP", "NSD", "UKBB"]
subdf["model"] = [k.split("_")[0] for k in subdf["key"]]
subdf["reg"] = [k.split("_")[0] for k in subdf["key"]]

,model,HCP,NSD,UKBB
0,attn_reg-1,0.868165,0.877245,0.809438
1,cross_reg-1,0.871352,0.878516,0.812113
2,crossreg_reg-16,0.876032,0.884056,0.816919
3,crossreg_reg-4,0.876681,0.884588,0.818250
4,crossreg_reg-1,0.881170,0.882289,0.823008
5,attn_pep-4,0.870163,0.879448,0.814755
6,cross_pep-4,0.873885,0.887200,0.816136
7,crossreg_reg-4_pep-4,0.877359,0.883790,0.820925


dataset,hcp,nsd,ukbb
pad,4,4,4
key,,,
attn_reg-1,0.868165,0.877245,0.809438
cross_reg-1,0.871352,0.878516,0.812113
crossreg_reg-16,0.876032,0.884056,0.816919
crossreg_reg-4,0.876681,0.884588,0.818250
crossreg_reg-1,0.881170,0.882289,0.823008
attn_pep-4,0.870163,0.879448,0.814755
cross_pep-4,0.873885,0.887200,0.816136
crossreg_reg-4_pep-4,0.877359,0.883790,0.820925


- nsd subj01 results

| model                |   loss (pad=0) |   loss (pad=4) |
|:---------------------|---------------:|---------------:|
| attn_reg-1           |         0.8336 |         0.8772 |
| attn_pep-4           |         0.8678 |         0.8794 |
| cross_reg-1          |         0.8357 |         0.8785 |
| cross_pep-4          |         0.8759 |         0.8872 |
| crossreg_reg-4       |         0.8721 |         0.8846 |
| crossreg_reg-4_pep-4 |         0.8757 |         0.8838 |

- ukbb val results

| model                |   loss (pad=0) |   loss (pad=4) |
|:---------------------|---------------:|---------------:|
| attn_reg-1           |         0.7461 |         0.8094 |
| attn_pep-4           |         0.8044 |         0.8148 |
| cross_reg-1          |         0.7497 |         0.8121 |
| cross_pep-4          |         0.8063 |         0.8161 |
| crossreg_reg-4       |         0.8075 |         0.8183 |
| crossreg_reg-4_pep-4 |         0.8147 |         0.8209 |